## Transformaciones: artefactos y su uso

Se construyen los tres artefactos que alimentan las Partes II–VI (**vectores de características**, **matrices de
interacción** y **grafos**), se explica para qué sirve cada uno y cómo el particionamiento afecta el plan de ejecución

- **Corre en:** Spark. Los self-joins de co-participación generan millones de pares; las técnicas O(n²) de las partes
  siguientes trabajan luego en local sobre estos artefactos ya reducidos

### Constantes

- `MAX_PART`: máximo de participantes de un proceso para generar pares de co-participación (≈ p90, `00_EDA/b_exploracion`)
- `MIN_COMP`: procesos compartidos mínimos para crear una arista de co-participación
- `HUB_PCT`: percentil de grado sobre el cual un proveedor se considera hub y sale del grafo filtrado
- `K_VECINOS`, `KNN_MIN_PROC`: vecinos por proveedor y procesos mínimos para entrar al grafo KNN
- `KNN_BUCKET`, `KNN_DIST`: ancho de bucket y distancia máxima (en desviaciones estándar) del LSH euclidiano

In [1]:
MAX_PART = 30
MIN_COMP = 2
HUB_PCT  = 0.99

K_VECINOS    = 10
KNN_MIN_PROC = 5
KNN_BUCKET   = 1.0
KNN_DIST     = 1.5

### Utilidades

In [2]:
NOTEBOOK = "c_uso"
%run ../funciones.ipynb
%run funciones.ipynb

In [3]:
spark = crear_spark(NOTEBOOK)
verificar_cluster(spark)

Master        : spark://spark-master:7077
Executors     : 2
Cores totales : 8
Memoria driver: 3.0 GB
UI del job    : http://localhost:4040


### Carga

In [4]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

hechos = leer(final_dir, "hechos").where(F.col("anio").isin(anios))
proc   = leer(final_dir, "procesos").where(F.col("anio").isin(anios))
part   = leer(final_dir, "participantes").where(F.col("anio").isin(anios))
dim_e, dim_p = leer(final_dir, "entidades"), leer(final_dir, "proveedores")

# nivel award × proveedor: deshace la explosión por familia
hp = (hechos.groupBy("ocid", "award_id", "proveedor_key", "entidad_key", "region", "categoria",
                     "postor_unico", "es_directa", "anio", "mes")
            .agg(F.sum("monto_asignado").alias("monto"))).cache()
print(f"award × proveedor: {hp.count():,}")

award × proveedor: 199,423


---
### 1. Vectores de características

#### a. Proveedor

In [5]:
def dependencia(df, actor, contraparte, sufijo):
    # share del principal cliente/proveedor y HHI de la distribución del monto
    w = Window.partitionBy(actor)
    return (df.groupBy(actor, contraparte).agg(F.sum("monto").alias("m"))
              .withColumn("s", F.col("m") / F.sum("m").over(w))
              .groupBy(actor).agg(F.max("s").alias(f"share_top_{sufijo}"), F.sum(F.col("s") ** 2).alias(f"hhi_{sufijo}"),
                                  F.count("*").alias(f"n_{sufijo}")))

cat = lambda v: F.avg((F.col("categoria") == v).cast("int"))
n_familias = lambda k: hechos.where("familia <> 'SIN'").groupBy(k).agg(F.countDistinct("familia").alias("n_familias"))

In [6]:
ofe_p = part.groupBy("proveedor_key").agg(F.sum("en_lista_oferentes").alias("n_ofertas"),
                                          F.sum(F.col("en_lista_oferentes") * F.col("gano")).alias("n_ganadas_compitiendo"))
feat_prov = (hp.groupBy("proveedor_key").agg(
        F.countDistinct("ocid").alias("n_procesos"), F.countDistinct("region").alias("n_regiones"),
        F.sum("monto").alias("monto_total"), F.expr("percentile_approx(monto, 0.5)").alias("monto_mediano"),
        F.avg("postor_unico").alias("pct_postor_unico"), F.avg("es_directa").alias("pct_directa"),
        cat("GOODS").alias("pct_bienes"), cat("SERVICES").alias("pct_servicios"), cat("WORKS").alias("pct_obras"),
        F.countDistinct("anio", "mes").alias("meses_activos"))
    .join(dependencia(hp, "proveedor_key", "entidad_key", "entidades"), "proveedor_key", "left")
    .join(n_familias("proveedor_key"), "proveedor_key", "left").join(ofe_p, "proveedor_key", "left")
    .join(dim_p.select("proveedor_key", "es_consorcio"), "proveedor_key", "left")
    .withColumn("compite", (F.coalesce("n_ofertas", F.lit(0)) > 0).cast("int"))
    .withColumn("tasa_exito", F.when(F.col("n_ofertas") > 0, F.col("n_ganadas_compitiendo") / F.col("n_ofertas")))
    .withColumn("log_n_procesos", F.log1p("n_procesos"))
    .withColumn("log_monto_total", F.log1p("monto_total")).withColumn("log_monto_mediano", F.log1p("monto_mediano")))

In [7]:
FEAT_PROV = ["log_n_procesos", "n_entidades", "n_regiones", "log_monto_total", "log_monto_mediano", "pct_postor_unico",
             "pct_directa", "pct_bienes", "pct_servicios", "pct_obras", "meses_activos", "share_top_entidades",
             "hhi_entidades", "n_familias", "compite", "tasa_exito", "es_consorcio"]
feat_prov = VectorAssembler(inputCols=FEAT_PROV, outputCol="features_raw").transform(feat_prov.fillna(0, FEAT_PROV))

#### b. Entidad

Las métricas de proceso salen de **todos** sus procesos, adjudicados o no

In [8]:
adjudicados = hp.select("ocid").distinct().withColumn("adjudicado", F.lit(1))
feat_ent = (proc.join(adjudicados, "ocid", "left").fillna({"adjudicado": 0})
    .groupBy("entidad_key").agg(
        F.count("*").alias("n_procesos"), F.avg("postor_unico").alias("pct_postor_unico"),
        F.avg("es_directa").alias("pct_directa"), F.avg("n_postores").alias("postores_promedio"),
        F.avg(1 - F.col("adjudicado")).alias("pct_sin_adjudicar"),
        cat("GOODS").alias("pct_bienes"), cat("SERVICES").alias("pct_servicios"), cat("WORKS").alias("pct_obras"))
    .join(hp.groupBy("entidad_key").agg(F.sum("monto").alias("monto_adjudicado_total"),
                                        F.expr("percentile_approx(monto, 0.5)").alias("monto_mediano")), "entidad_key", "left")
    .join(dependencia(hp, "entidad_key", "proveedor_key", "proveedores"), "entidad_key", "left")
    .join(n_familias("entidad_key"), "entidad_key", "left")
    .join(dim_e.select("entidad_key", "region"), "entidad_key", "left")
    .withColumn("log_n_procesos", F.log1p("n_procesos"))
    .withColumn("log_monto_total", F.log1p("monto_adjudicado_total")).withColumn("log_monto_mediano", F.log1p("monto_mediano")))

FEAT_ENT = ["log_n_procesos", "pct_postor_unico", "pct_directa", "postores_promedio", "pct_sin_adjudicar", "pct_bienes",
            "pct_servicios", "pct_obras", "log_monto_total", "log_monto_mediano", "share_top_proveedores",
            "hhi_proveedores", "n_proveedores", "n_familias"]
feat_ent = VectorAssembler(inputCols=FEAT_ENT, outputCol="features_raw").transform(feat_ent.fillna(0, FEAT_ENT))

- **Uso (Parte II):** KMeans++, DBSCAN/HDBSCAN y jerárquico; tipologías de actores y outliers (PQ1, PQ2)
- Se guardan **sin estandarizar**: la estandarización y el PCA se justifican en la Parte II
- `log1p` en montos y conteos por la cola pesada (`00_EDA/a_entendimiento` §3.f)
- `share_top_*` y `hhi_*` miden dependencia: HHI = Σ (participación de cada contraparte)², 1 = una sola contraparte
- `compite` separa "nunca oferta" de "oferta y nunca gana", que el `fillna(0)` de `tasa_exito` igualaría
- `es_consorcio`: el consorcio es un nodo propio, porque la fuente no publica sus miembros

---
### 2. Matrices de interacción

Formato largo (COO): una fila por celda no vacía, con índices enteros para ALS y SVD, y **una fila por año**

In [9]:
def indexar(df, entrada, salida):
    idx = StringIndexer(inputCol=entrada, outputCol=salida).fit(df)
    return idx, idx.labelsArray[0]

def a_enteros(df, cols):
    for c in cols:
        df = df.withColumn(c, F.col(c).cast("int"))
    return df

ep = (hp.groupBy("entidad_key", "proveedor_key", "anio")
        .agg(F.count("*").alias("n_adj"), F.sum("monto").alias("monto")))
ef = (hechos.where("familia <> 'SIN'").groupBy("entidad_key", "familia", "anio")
            .agg(F.countDistinct("ocid").alias("n_procesos"), F.sum("monto_asignado").alias("monto")))
idx_e, lab_e = indexar(ep.select("entidad_key").unionByName(ef.select("entidad_key")), "entidad_key", "entidad_idx")
idx_p, lab_p = indexar(ep, "proveedor_key", "proveedor_idx")
idx_f, lab_f = indexar(ef, "familia", "familia_idx")

In [10]:
m_ep = (a_enteros(idx_p.transform(idx_e.transform(ep)), ["entidad_idx", "proveedor_idx"])
          .withColumn("senal_log_monto", F.log1p(F.coalesce("monto", F.lit(0)))))
m_ef = a_enteros(idx_f.transform(idx_e.transform(ef)), ["entidad_idx", "familia_idx"])
indices = {k: spark.createDataFrame(list(enumerate(v)), ["idx", "clave"])
           for k, v in {"indice_entidad": lab_e, "indice_proveedor": lab_p, "indice_familia": lab_f}.items()}

def densidad(m, fila, columna, n_f, n_c):
    nnz = m.select(fila, columna).distinct().count()
    return nnz, nnz / (n_f * n_c)

n_e, n_p, n_f = len(lab_e), len(lab_p), len(lab_f)
(nnz_ep, d_ep), (nnz_ef, d_ef) = densidad(m_ep, "entidad_idx", "proveedor_idx", n_e, n_p), densidad(m_ef, "entidad_idx", "familia_idx", n_e, n_f)
print(f"entidad × proveedor: {n_e:,} × {n_p:,} | pares {nnz_ep:,} | densidad {d_ep:.4%}")
print(f"entidad × familia  : {n_e:,} × {n_f:,} | pares {nnz_ef:,} | densidad {d_ef:.2%}")

entidad × proveedor: 2,992 × 86,032 | pares 136,124 | densidad 0.0529%
entidad × familia  : 2,992 × 386 | pares 39,349 | densidad 3.41%


- **Uso (Parte III):**
  - `matriz_entidad_proveedor`: ALS implícito y filtrado por vecindario (¿a quién le compra?)
  - `matriz_entidad_familia`: perfil de rubros (¿qué compra?), que sirve para entidades parecidas, *cold start* y para
    distinguir sustituto de rubro nuevo
- **Uso (Parte VI):** SVD truncada de la matriz entidad × proveedor como embedding de entidades y proveedores
- **Señal implícita:** `n_adj` / `n_procesos` (frecuencia) o `senal_log_monto`; la Parte III elige cuál usar como confianza
- **`anio` en cada fila:** el hold-out temporal (entrenar ≤ 2024, evaluar 2025) agrega solo los años de cada lado, **sin
  fuga** de 2025 al entrenamiento
- `StringIndexer` asigna índices de forma distribuida (un `row_number()` sin partición llevaría todo a una sola tarea);
  `indice_*` permite volver del entero a la clave

---
### 3. Grafos

#### a. Bipartito entidad → proveedor

In [11]:
e_p = hp.groupBy("entidad_key", "proveedor_key").agg(F.count("*").alias("n_adj"), F.sum("monto").alias("monto"),
                                                    F.min("anio").alias("anio_min"), F.max("anio").alias("anio_max"))
nodo = lambda df, k, pref, tipo, cols: df.select(F.concat(F.lit(pref), k).alias("id"), F.lit(tipo).alias("tipo"), *cols)
v_ent = nodo(dim_e.join(e_p.select("entidad_key").distinct(), "entidad_key"), "entidad_key", "E|", "entidad", ["nombre", "region"])
v_prov = nodo(dim_p.join(e_p.select("proveedor_key").distinct(), "proveedor_key"), "proveedor_key", "P|", "proveedor",
              ["nombre", "region", "es_consorcio"])
vertices_bip = v_ent.unionByName(v_prov, allowMissingColumns=True)
aristas_bip = e_p.select(F.concat(F.lit("E|"), "entidad_key").alias("src"), F.concat(F.lit("P|"), "proveedor_key").alias("dst"),
                         "n_adj", "monto", "anio_min", "anio_max")

- **Nodos:** entidades (`E|`) y proveedores (`P|`); el prefijo evita fundir una entidad pública que también es proveedor
- **Arista dirigida** entidad → proveedor si hubo al menos una adjudicación; peso `n_adj` o `monto`
- **Uso (Parte IV):** PageRank (GraphFrames) e HITS: hubs = entidades que compran a proveedores centrales, authorities =
  proveedores a los que compran buenos hubs (PQ1)

#### b. Co-participación proveedor — proveedor

In [12]:
SHUFFLE = int(spark.conf.get("spark.sql.shuffle.partitions"))

n_part = part.groupBy("ocid").agg(F.count("*").alias("n"))
part_n = (part.select("ocid", "proveedor_key").join(n_part.where(f"n BETWEEN 2 AND {MAX_PART}"), "ocid")
              .repartition(SHUFFLE, "ocid").cache())
pares = (part_n.alias("a").join(part_n.alias("b"),
            (F.col("a.ocid") == F.col("b.ocid")) & (F.col("a.proveedor_key") < F.col("b.proveedor_key")))
         .select(F.col("a.proveedor_key").alias("src"), F.col("b.proveedor_key").alias("dst"), F.col("a.n").alias("n")))
aristas_co = (pares.groupBy("src", "dst").agg(F.count("*").alias("n_coparticipa"), F.sum(1.0 / (F.col("n") - 1)).alias("peso"))
              .where(f"n_coparticipa >= {MIN_COMP}")
              .withColumn("src", F.concat(F.lit("P|"), "src")).withColumn("dst", F.concat(F.lit("P|"), "dst"))).cache()

In [13]:
extremos = aristas_co.select(F.col("src").alias("id")).unionByName(aristas_co.select(F.col("dst").alias("id")))
vertices_co = nodo(dim_p, "proveedor_key", "P|", "proveedor", ["nombre", "region", "es_consorcio"]).join(extremos.distinct(), "id")
grado_co = extremos.groupBy("id").count()
q_grado = grado_co.approxQuantile("count", [.5, .9, .99], .01)
print(f"co-participación: {vertices_co.count():,} nodos | {aristas_co.count():,} aristas | grado p50/p90/p99: {q_grado}")
(grado_co.orderBy(F.desc("count")).limit(5).join(vertices_co.select("id", "nombre"), "id").orderBy(F.desc("count"))
    .show(truncate=50))

co-participación: 60,230 nodos | 1,640,718 aristas | grado p50/p90/p99: [16.0, 113.0, 12059.0]


+-----------------+-----+---------------------------------------+
|               id|count|                                 nombre|
+-----------------+-----+---------------------------------------+
|P|RUC:20611499591|12059|              TVT QONTRATISTAS E.I.R.L.|
|P|RUC:20605753907| 9522|INVERSIONES GENERALES LAURA ROCA S.A.C.|
|P|RUC:20542513439| 7861|      CRISAMYR SOCIEDAD ANONIMA CERRADA|
|P|RUC:20612573621| 7168|               CONSTRUCAPRIGLE E.I.R.L.|
|P|RUC:20542059291| 6523|         CORPORACION GRUPO IBIZA S.A.C.|
+-----------------+-----+---------------------------------------+



- **Criterio de arista:** dos proveedores se conectan si participaron (ofertaron o ganaron) juntos en **≥ 2 procesos** de
  entre 2 y `MAX_PART` participantes
- **Peso** = Σ 1/(n−1) (ponderación de Newman): coincidir en un proceso de 3 pesa 0.5 y en uno de 30, 0.03
- **Tope y umbral:** un proceso de n participantes genera n(n−1)/2 pares, así que los masivos dominarían el grafo con
  vínculos débiles; una sola coincidencia es mayormente casualidad
- **Uso (Partes V y VI):** comunidades (Girvan-Newman/espectral en subgrafo, BigCLAM) y Node2Vec (PQ2)
- **Grafo KNN:** se construye igual como comparación (§3.d); la co-participación sigue siendo el grafo principal porque sus aristas son hechos observados, no similitudes

#### c. Co-participación sin hubs

In [14]:
umbral_hub = grado_co.approxQuantile("count", [HUB_PCT], .001)[0]
hubs = grado_co.where(F.col("count") > umbral_hub).select("id")
aristas_co_sh = (aristas_co.join(hubs.withColumnRenamed("id", "src"), "src", "left_anti")
                           .join(hubs.withColumnRenamed("id", "dst"), "dst", "left_anti")).cache()
ext_sh = aristas_co_sh.select(F.col("src").alias("id")).unionByName(aristas_co_sh.select(F.col("dst").alias("id"))).distinct()
vertices_co_sh = vertices_co.join(ext_sh, "id")
print(f"umbral de hub: grado > {umbral_hub:,.0f} | hubs: {hubs.count():,} | "
      f"sin hubs: {vertices_co_sh.count():,} nodos, {aristas_co_sh.count():,} aristas "
      f"({aristas_co_sh.count() / aristas_co.count():.1%} de las aristas)")

umbral de hub: grado > 527 | hubs: 662 | sin hubs: 57,144 nodos, 896,421 aristas (54.6% de las aristas)


- **Criterio:** se quitan los proveedores con grado sobre el p99 (`HUB_PCT`) y todas sus aristas; los nodos que quedan
  aislados también salen
- **Por qué:** unas pocas constructoras participan en miles de procesos y se conectan con casi todos; en
  Girvan-Newman, espectral y Node2Vec fusionarían todas las comunidades en una sola (los caminos aleatorios pasan siempre
  por ellos)
- **Se conservan aparte:** el grafo completo sigue en `grafo_coparticipacion`; los hubs se analizan como actores
  centrales en la Parte IV, no se descartan del estudio
- **Uso (Partes V y VI):** comunidades y Node2Vec trabajan sobre esta versión
- **Resultado:** el p99 de grado es 527; salen 662 hubs (1.1% de los nodos) pero se lleva el **45.4% de las aristas**
  (1.64M → 896k): la concentración de la co-participación en pocos actores es extrema

#### d. Grafo KNN sobre vectores de proveedor

- Cada proveedor se conecta a sus `K_VECINOS` más cercanos en `features_proveedor` **estandarizado**
- **LSH euclidiano** (`BucketedRandomProjectionLSH`) para no comparar todos contra todos: solo pares que caen en un mismo
  bucket y a distancia ≤ `KNN_DIST`
- Solo proveedores con ≥ `KNN_MIN_PROC` procesos: con 1 o 2 procesos los vectores son casi idénticos y generan
  vecindarios enormes sin información

In [15]:
from pyspark.ml.feature import StandardScaler, BucketedRandomProjectionLSH

base_knn = feat_prov.where(f"n_procesos >= {KNN_MIN_PROC}").select(
    F.concat(F.lit("P|"), "proveedor_key").alias("id"), "features_raw")
esc = (StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)
       .fit(base_knn).transform(base_knn).select("id", "features").repartition(SHUFFLE).cache())
brp = BucketedRandomProjectionLSH(inputCol="features", outputCol="hashes", bucketLength=KNN_BUCKET,
                                  numHashTables=3, seed=42).fit(esc)
print(f"proveedores en KNN: {esc.count():,}")

proveedores en KNN: 7,100


In [16]:
cand_knn = (brp.approxSimilarityJoin(esc, esc, KNN_DIST, distCol="distancia")
               .select(F.col("datasetA.id").alias("src"), F.col("datasetB.id").alias("dst"), "distancia")
               .where("src <> dst"))
w = Window.partitionBy("src").orderBy("distancia", "dst")
aristas_knn = materializar(cand_knn.withColumn("rango", F.row_number().over(w)).where(f"rango <= {K_VECINOS}"), "aristas_knn")
vertices_knn = nodo(dim_p, "proveedor_key", "P|", "proveedor", ["nombre", "region", "es_consorcio"]).join(esc.select("id"), "id")

vecinos = aristas_knn.groupBy("src").count()
print(f"KNN: {aristas_knn.count():,} aristas dirigidas | proveedores sin vecinos a distancia ≤ {KNN_DIST}: "
      f"{esc.count() - vecinos.count():,} | con menos de {K_VECINOS}: {vecinos.where(f'count < {K_VECINOS}').count():,}")

KNN: 56,848 aristas dirigidas | proveedores sin vecinos a distancia ≤ 1.5: 594 | con menos de 10: 1,314


In [17]:
# comparación con co-participación, como grafos no dirigidos
no_dirigido = lambda a: a.select(F.least("src", "dst").alias("src"), F.greatest("src", "dst").alias("dst")).distinct()
knn_u = no_dirigido(aristas_knn).cache()
co_en_knn = aristas_co.join(esc.select(F.col("id").alias("src")), "src").join(esc.select(F.col("id").alias("dst")), "dst")

def resumen_grafo(nombre, v, a):
    n, m = v.count(), a.count()
    return (nombre, n, m, 2 * m / n if n else 0)

comparacion = pd.DataFrame([resumen_grafo("co-participación", vertices_co, aristas_co),
                            resumen_grafo("co-participación sin hubs", vertices_co_sh, aristas_co_sh),
                            resumen_grafo("KNN (no dirigido)", vertices_knn, knn_u)],
                           columns=["grafo", "nodos", "aristas", "grado_medio"])
comunes = knn_u.join(co_en_knn.select("src", "dst"), ["src", "dst"]).count()
print(f"aristas KNN que también son co-participación: {comunes:,} de {knn_u.count():,} ({comunes / knn_u.count():.1%})")
print(f"aristas de co-participación entre nodos KNN que KNN recupera: {comunes / co_en_knn.count():.1%}")
comparacion

aristas KNN que también son co-participación: 2,286 de 38,394 (6.0%)


aristas de co-participación entre nodos KNN que KNN recupera: 1.0%


,grafo,nodos,aristas,grado_medio
0,co-participación,60230,1640718,54.481753
1,co-participación sin hubs,57144,896421,31.374108
2,KNN (no dirigido),7100,38394,10.815211


- **Criterio de arista KNN:** `src → dst` si `dst` está entre los `K_VECINOS` más cercanos de `src` (distancia euclidiana
  sobre features estandarizados); peso = `distancia`, orden = `rango`. Dirigido: la relación de vecindad no es simétrica
- **Resultado:** 7,100 proveedores, 38,394 aristas no dirigidas; 594 quedan sin vecinos a distancia ≤ 1.5 (perfiles
  atípicos, candidatos a outlier en la Parte II)
- **Solapamiento casi nulo:** solo el 6.0% de las aristas KNN son también co-participación, y KNN recupera el 1.0% de la
  co-participación entre esos nodos. **Parecerse** (perfil de montos, rubros, competencia) no implica **competir** en
  los mismos procesos: las dos vistas son complementarias, no redundantes
- **Ajuste:** si muchos proveedores quedan sin vecinos, subir `KNN_DIST` o `KNN_BUCKET`; si el join es muy pesado,
  bajarlos o subir `KNN_MIN_PROC`
- **Uso (Parte VI):** comparar embeddings de Node2Vec sobre co-participación con los de este grafo

---
### 4. Particionamiento y plan de ejecución

#### a. Self-join de co-participación

In [18]:
import requests

def tiempo(df, grupo=None):
    if grupo:
        spark.sparkContext.setJobGroup(grupo, grupo)
    t0 = time.time(); n = df.count(); t = round(time.time() - t0, 1)
    spark.sparkContext.setLocalProperty("spark.jobGroup.id", None)
    return n, t

def metricas_stages(grupo):
    # lee de la API REST de Spark UI los stages de los jobs del grupo (lo mismo que muestra localhost:4040)
    sc = spark.sparkContext
    api = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}"
    jobs = [j for j in requests.get(f"{api}/jobs").json() if j.get("jobGroup") == grupo]
    st = [s for j in jobs for sid in j["stageIds"] for s in requests.get(f"{api}/stages/{sid}").json()[:1]]
    hechos_ = [s for s in st if s["status"] == "COMPLETE"]
    mb = lambda k: round(sum(s[k] for s in hechos_) / 2**20, 1)
    return {"corrida": grupo, "jobs": len(jobs), "stages": len(hechos_),
            "stages_omitidos": sum(s["status"] == "SKIPPED" for s in st), "tareas": sum(s["numTasks"] for s in hechos_),
            "lectura_MB": mb("inputBytes"), "shuffle_escrito_MB": mb("shuffleWriteBytes"),
            "shuffle_leido_MB": mb("shuffleReadBytes")}

part_sin = part.select("ocid", "proveedor_key").join(n_part.where(f"n BETWEEN 2 AND {MAX_PART}"), "ocid")
pares_sin = (part_sin.alias("a").join(part_sin.alias("b"),
                (F.col("a.ocid") == F.col("b.ocid")) & (F.col("a.proveedor_key") < F.col("b.proveedor_key"))))
pares_sin.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [ocid#80], [ocid#9847], Inner, (proveedor_key#81 < proveedor_key#9848)
   :- Project [ocid#80, proveedor_key#81, n#2536L]
   :  +- SortMergeJoin [ocid#80], [ocid#9837], Inner
   :     :- Sort [ocid#80 ASC NULLS FIRST], false, 0
   :     :  +- Exchange hashpartitioning(ocid#80, 24), ENSURE_REQUIREMENTS, [plan_id=10582]
   :     :     +- Project [ocid#80, proveedor_key#81]
   :     :        +- Filter (isnotnull(ocid#80) AND isnotnull(proveedor_key#81))
   :     :           +- FileScan parquet [ocid#80,proveedor_key#81,anio#84,mes#85] Batched: true, DataFilters: [isnotnull(ocid#80), isnotnull(proveedor_key#81)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/final/participantes], PartitionFilters: [anio#84 IN (2023,2024,2025)], PushedFilters: [IsNotNull(ocid), IsNotNull(proveedor_key)], ReadSchema: struct<ocid:string,proveedor_key:string>
   :     +- Sort [ocid#9837 ASC NULLS FIRST]

In [19]:
pares.explain()
print("sin repartition:", tiempo(pares_sin, "sin_repartition"),
      "| con repartition(ocid) + cache:", tiempo(pares, "con_repartition"))
dag = pd.DataFrame([metricas_stages("sin_repartition"), metricas_stages("con_repartition")])
dag

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [proveedor_key#81 AS src#2595, proveedor_key#2569 AS dst#2596, n#2536L]
   +- SortMergeJoin [ocid#80], [ocid#2568], Inner, (proveedor_key#81 < proveedor_key#2569)
      :- Filter (isnotnull(ocid#80) AND isnotnull(proveedor_key#81))
      :  +- InMemoryTableScan [ocid#80, proveedor_key#81, n#2536L], [isnotnull(ocid#80), isnotnull(proveedor_key#81)]
      :        +- InMemoryRelation [ocid#80, proveedor_key#81, n#2536L], StorageLevel(disk, memory, deserialized, 1 replicas)
      :              +- *(5) Project [ocid#80, proveedor_key#81, n#2536L]
      :                 +- *(5) SortMergeJoin [ocid#80], [ocid#2541], Inner
      :                    :- *(2) Sort [ocid#80 ASC NULLS FIRST], false, 0
      :                    :  +- Exchange hashpartitioning(ocid#80, 24), ENSURE_REQUIREMENTS, [plan_id=10626]
      :                    :     +- *(1) Project [ocid#80, proveedor_key#81]
      :                    :        +- *(1) 

sin repartition: (13097305, 32.2) | con repartition(ocid) + cache: (13097305, 0.4)


,corrida,jobs,stages,stages_omitidos,tareas,lectura_MB,shuffle_escrito_MB,shuffle_leido_MB
0,sin_repartition,5,5,4,69,58.5,43.1,83.3
1,con_repartition,2,2,5,25,61.2,0.0,0.0


- **Sin reparticionar:** cada lado del self-join vuelve a leer `participantes` desde disco y recalcula el conteo por
  proceso (`FileScan` + `Exchange hashpartitioning(ocid)` en cada rama); AQE termina eligiendo `BroadcastHashJoin`
- **Con `repartition(ocid)` + `cache`:** ambos lados leen `InMemoryTableScan` ya particionado por `ocid` y el
  `SortMergeJoin` no necesita otro shuffle
- Parte de la mejora en tiempo es el cache (los datos ya están en memoria), pero el DAG muestra que además se elimina
  el shuffle repetido
- **DAG (Spark UI, vía API REST):** sin reparticionar, 5 stages y 69 tareas escriben 43.1 MB y leen 83.3 MB de shuffle;
  con `repartition` + `cache`, 2 stages y 25 tareas con **0 MB de shuffle**: los 5 stages del shuffle por `ocid`
  aparecen como omitidos (`SKIPPED`) porque Spark reutiliza el ya hecho
- **Tiempo (cluster de 2 workers × 4 cores):** 32.2 s → 0.4 s para los mismos 13.1M pares

#### b. Broadcast y partition pruning

In [20]:
hechos.join(F.broadcast(dim_e), "entidad_key").explain()
leer(final_dir, "hechos").where("anio = 2025 AND mes = 3").select("ocid").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [entidad_key#3, ocid#0, award_id#1, fecha#2, proveedor_key#4, region#5, categoria#6, metodo_detalle#7, postor_unico#8, es_directa#9, n_postores#10, n_proveedores_adj#11L, tiene_contrato#12, monto_adj#13, monto_asignado#14, familia#15, segmento#16, anio#17, mes#18, nombre#94, ruc#95, region#96]
   +- BroadcastHashJoin [entidad_key#3], [entidad_key#93], Inner, BuildRight, false
      :- Filter isnotnull(entidad_key#3)
      :  +- FileScan parquet [ocid#0,award_id#1,fecha#2,entidad_key#3,proveedor_key#4,region#5,categoria#6,metodo_detalle#7,postor_unico#8,es_directa#9,n_postores#10,n_proveedores_adj#11L,tiene_contrato#12,monto_adj#13,monto_asignado#14,familia#15,segmento#16,anio#17,mes#18] Batched: true, DataFilters: [isnotnull(entidad_key#3)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/final/hechos], PartitionFilters: [anio#17 IN (2023,2024,2025)], PushedFilters: [IsNotNull(entidad_k

== Physical Plan ==
*(1) Project [ocid#10155]
+- *(1) ColumnarToRow
   +- FileScan parquet [ocid#10155,anio#10172,mes#10173] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/final/hechos], PartitionFilters: [isnotnull(anio#10172), isnotnull(mes#10173), (anio#10172 = 2025), (mes#10173 = 3)], PushedFilters: [], ReadSchema: struct<ocid:string>




- **Dimensión chica (entidades):** `BroadcastHashJoin`. La dimensión se copia a cada executor y la tabla de hechos no se
  mueve (sin `Exchange`)
- **Partition pruning:** el filtro `anio = 2025 AND mes = 3` aparece en `PartitionFilters`: Spark lee solo esa carpeta de
  `hechos` (1 de 36 meses), sin escanear el resto. Es lo que hace barato el hold-out temporal

#### c. Sesgo

In [21]:
(aristas_co.groupBy(F.spark_partition_id().alias("particion")).count()
    .agg(F.min("count").alias("min"), F.expr("percentile_approx(count, 0.5)").alias("mediana"), F.max("count").alias("max")).show())

+-----+-------+-----+
|  min|mediana|  max|
+-----+-------+-----+
|67802|  68340|68947|
+-----+-------+-----+



- Las aristas de co-participación quedan **balanceadas** entre particiones (67,766 a 68,907 filas): el hash sobre
  `(src, dst)` reparte bien
- El sesgo está en los **nodos**, no en las particiones: unos pocos proveedores tienen grado > 12,000 (constructoras que
  participan en miles de procesos). La Parte V deberá tratarlos (filtrar hubs o normalizar) antes de buscar comunidades
- AQE (activo en Spark 3.5) divide particiones sesgadas en los joins; se puede verificar en `localhost:4040`

---
### 5. Exportación de artefactos (`data/artefactos/`)

In [22]:
artefactos = {
    "features_proveedor": feat_prov, "features_entidad": feat_ent,
    "matriz_entidad_proveedor": m_ep, "matriz_entidad_familia": m_ef,
    **{k: v.coalesce(1) for k, v in indices.items()},
    "grafo_bipartito/vertices": vertices_bip, "grafo_bipartito/aristas": aristas_bip,
    "grafo_coparticipacion/vertices": vertices_co, "grafo_coparticipacion/aristas": aristas_co,
    "grafo_coparticipacion_sin_hubs/vertices": vertices_co_sh, "grafo_coparticipacion_sin_hubs/aristas": aristas_co_sh,
    "grafo_knn/vertices": vertices_knn, "grafo_knn/aristas": aristas_knn}
for nombre, df in artefactos.items():
    df.write.mode("overwrite").parquet(str(artef_dir / nombre))
print("artefactos:", ", ".join(artefactos))

artefactos: features_proveedor, features_entidad, matriz_entidad_proveedor, matriz_entidad_familia, indice_entidad, indice_proveedor, indice_familia, grafo_bipartito/vertices, grafo_bipartito/aristas, grafo_coparticipacion/vertices, grafo_coparticipacion/aristas, grafo_coparticipacion_sin_hubs/vertices, grafo_coparticipacion_sin_hubs/aristas, grafo_knn/vertices, grafo_knn/aristas


| Artefacto | Grano | Parte | Técnica | Pregunta |
|---|---|---|---|---|
| `features_proveedor` | proveedor | II | KMeans++, DBSCAN/HDBSCAN, jerárquico | PQ1, PQ2 |
| `features_entidad` | entidad | II | ídem; entidades cautivas | PQ1 |
| `matriz_entidad_proveedor` | entidad × proveedor × año | III, VI | ALS implícito, vecindario, SVD | PQ3 |
| `matriz_entidad_familia` | entidad × familia × año | III | perfil de rubros, *cold start* | PQ3 |
| `indice_*` | índice → clave | III, VI | traducir resultados | — |
| `grafo_bipartito` | entidad → proveedor | IV | PageRank, HITS | PQ1 |
| `grafo_coparticipacion` | proveedor — proveedor | V | Louvain/LPA sobre el grafo completo (escalabilidad) | PQ2 |
| `grafo_coparticipacion_sin_hubs` | proveedor — proveedor | V, VI | Girvan-Newman/espectral, BigCLAM, Node2Vec | PQ2 |
| `grafo_knn` | proveedor → k vecinos | VI | comparación de embeddings | PQ2 |

#### a. Validación de integridad

- Se valida lo **escrito en disco** (no los DataFrames en memoria): es lo que leen las Partes II a VII
- Cada regla cuenta las filas que la violan; **0 = OK**. Cualquier falla detiene el notebook

In [23]:
art = {n: leer(artef_dir, n) for n in artefactos}
reglas = []

def regla(artefacto, descripcion, violaciones):
    reglas.append((artefacto, descripcion, violaciones if isinstance(violaciones, int) else violaciones.count()))

def duplicados(df, cols):
    return df.groupBy(*cols).count().where("count > 1")

def nulos(df, cols):
    return df.where(" OR ".join(f"`{c}` IS NULL OR isnan(`{c}`)" for c in cols))

In [24]:
# vectores de características
for n, k, cols in (("features_proveedor", "proveedor_key", FEAT_PROV), ("features_entidad", "entidad_key", FEAT_ENT)):
    df = art[n]
    regla(n, f"{k} nulo", df.where(F.col(k).isNull()))
    regla(n, f"{k} duplicado", duplicados(df, [k]))
    regla(n, "feature nula o NaN", nulos(df, cols))
regla("features_proveedor", "proveedor de hechos sin vector",
      hp.select("proveedor_key").distinct().join(art["features_proveedor"], "proveedor_key", "left_anti"))
regla("features_proveedor", "share/HHI fuera de [0, 1]",
      art["features_proveedor"].where("share_top_entidades NOT BETWEEN 0 AND 1.000001 OR hhi_entidades NOT BETWEEN 0 AND 1.000001"))

In [25]:
# índices: enteros contiguos 0..n-1 y claves únicas
for n in ("indice_entidad", "indice_proveedor", "indice_familia"):
    r = art[n].agg(F.count("*").alias("n"), F.countDistinct("idx").alias("ni"), F.countDistinct("clave").alias("nc"),
                   F.min("idx").alias("lo"), F.max("idx").alias("hi")).first()
    regla(n, "idx no contiguo desde 0", int(not (r.lo == 0 and r.hi == r.n - 1 and r.ni == r.n)))
    regla(n, "clave duplicada", r.n - r.nc)

# matrices COO: índices válidos, celdas únicas, señal positiva
for n, col, idx, senal in (("matriz_entidad_proveedor", "proveedor_idx", "indice_proveedor", "n_adj"),
                            ("matriz_entidad_familia", "familia_idx", "indice_familia", "n_procesos")):
    m = art[n]
    regla(n, "celda (fila, columna, anio) duplicada", duplicados(m, ["entidad_idx", col, "anio"]))
    regla(n, "entidad_idx fuera del índice",
          m.join(art["indice_entidad"], m.entidad_idx == art["indice_entidad"].idx, "left_anti"))
    regla(n, f"{col} fuera del índice", m.join(art[idx], m[col] == art[idx].idx, "left_anti"))
    regla(n, f"{senal} <= 0 o monto < 0", m.where(f"{senal} <= 0 OR monto < 0"))

In [26]:
# grafos: aristas con extremos existentes, sin duplicados ni lazos
for g in ("grafo_bipartito", "grafo_coparticipacion", "grafo_coparticipacion_sin_hubs", "grafo_knn"):
    v, a = art[f"{g}/vertices"], art[f"{g}/aristas"]
    regla(g, "vértice duplicado", duplicados(v, ["id"]))
    regla(g, "arista duplicada", duplicados(a, ["src", "dst"]))
    regla(g, "lazo (src = dst)", a.where("src = dst"))
    regla(g, "src sin vértice", a.join(v, a.src == v.id, "left_anti"))
    regla(g, "dst sin vértice", a.join(v, a.dst == v.id, "left_anti"))

bip, co = art["grafo_bipartito/aristas"], art["grafo_coparticipacion/aristas"]
regla("grafo_bipartito", "arista que no va E| → P|", bip.where("NOT (src LIKE 'E|%' AND dst LIKE 'P|%')"))
regla("grafo_coparticipacion", "no dirigida: falta src < dst", co.where("src >= dst"))
co_sh = art["grafo_coparticipacion_sin_hubs/aristas"]
regla("grafo_coparticipacion_sin_hubs", "arista que no está en el grafo completo", co_sh.join(co, ["src", "dst"], "left_anti"))
regla("grafo_coparticipacion_sin_hubs", "arista que toca un hub",
      co_sh.join(hubs.withColumnRenamed("id", "src"), "src").unionByName(co_sh.join(hubs.withColumnRenamed("id", "dst"), "dst")))
knn = art["grafo_knn/aristas"]
regla("grafo_knn", f"más de {K_VECINOS} vecinos", knn.groupBy("src").count().where(f"count > {K_VECINOS}"))
regla("grafo_knn", f"distancia > {KNN_DIST}", knn.where(f"distancia > {KNN_DIST}"))
regla("grafo_coparticipacion", f"n_coparticipa < {MIN_COMP} o peso <= 0", co.where(f"n_coparticipa < {MIN_COMP} OR peso <= 0"))

In [27]:
# consistencia entre artefactos: bipartito y matriz entidad × proveedor salen de los mismos hechos
tot = lambda df: df.agg(F.sum("n_adj").alias("n"), F.sum("monto").alias("m")).first()
t_bip, t_ep = tot(bip), tot(art["matriz_entidad_proveedor"])
regla("bipartito vs. matriz", "Σ n_adj distinta", abs(t_bip.n - t_ep.n))
regla("bipartito vs. matriz", "Σ monto difiere > 0.01%", int(abs(t_bip.m - t_ep.m) > 1e-4 * t_ep.m))
regla("bipartito vs. matriz", "aristas ≠ pares entidad × proveedor", abs(bip.count() - nnz_ep))

validacion = pd.DataFrame(reglas, columns=["artefacto", "regla", "violaciones"])
validacion["ok"] = validacion.violaciones == 0
fallas = validacion[~validacion.ok]
assert fallas.empty, f"reglas violadas:\n{fallas.to_string(index=False)}"
print(f"OK: {len(validacion)} reglas sobre {validacion.artefacto.nunique()} artefactos")
validacion

OK: 52 reglas sobre 12 artefactos


,artefacto,regla,violaciones,ok
0,features_proveedor,proveedor_key nulo,0,True
1,features_proveedor,proveedor_key duplicado,0,True
2,features_proveedor,feature nula o NaN,0,True
3,features_entidad,entidad_key nulo,0,True
4,features_entidad,entidad_key duplicado,0,True
5,features_entidad,feature nula o NaN,0,True
6,features_proveedor,proveedor de hechos sin vector,0,True
7,features_proveedor,"share/HHI fuera de [0, 1]",0,True
8,indice_entidad,idx no contiguo desde 0,0,True
9,indice_entidad,clave duplicada,0,True


- **Unicidad:** una fila por actor en los vectores, una por celda en las matrices, una por par en los grafos
- **Integridad referencial:** todo índice de matriz existe en `indice_*`; toda arista apunta a vértices existentes
- **Semántica:** el bipartito va siempre entidad → proveedor; la co-participación se guarda una vez por par (`src < dst`)
- **Cuadre:** el bipartito y la matriz entidad × proveedor suman las mismas adjudicaciones y el mismo monto

#### b. Particiones físicas

In [28]:
def fisico(nombre):
    archivos = list((artef_dir / nombre).rglob("*.parquet"))
    mb = sum(a.stat().st_size for a in archivos) / 2**20
    return (nombre, artefactos[nombre].rdd.getNumPartitions(), len(archivos), round(mb, 2),
            round(mb / len(archivos), 2) if archivos else 0)

particiones = pd.DataFrame([fisico(n) for n in artefactos],
                           columns=["artefacto", "particiones", "archivos", "MB", "MB_por_archivo"])
particiones

,artefacto,particiones,archivos,MB,MB_por_archivo
0,features_proveedor,10,10,5.67,0.57
1,features_entidad,1,1,0.45,0.45
2,matriz_entidad_proveedor,6,6,4.59,0.77
3,matriz_entidad_familia,2,2,0.79,0.40
4,indice_entidad,1,1,0.03,0.03
5,indice_proveedor,1,1,0.85,0.85
6,indice_familia,1,1,0.00,0.00
7,grafo_bipartito/vertices,3,3,2.23,0.74
8,grafo_bipartito/aristas,6,6,2.28,0.38
9,grafo_coparticipacion/vertices,9,9,1.72,0.19


- **Una partición = un archivo:** al escribir, cada tarea produce su propio `part-*.parquet`
- **Índices con `coalesce(1)`:** son tablas chicas (una fila por clave); en varios archivos solo agregarían overhead de
  apertura, y `coalesce` no hace shuffle (solo junta particiones)
- **Resto sin `coalesce`:** se escriben con las particiones de su último shuffle (`spark.sql.shuffle.partitions` = 24);
  así la lectura en las Partes II a VI arranca con 24 tareas en paralelo
- **Archivos de 0.1 a 1 MB:** todos los artefactos son chicos (el mayor, co-participación, 22.9 MB en 24 archivos). Es
  *small files* moderado; se mantiene porque la lectura paralela compensa y los artefactos se leen pocas veces. Con más
  años de datos convendría `coalesce` a archivos de ~128 MB

---
### 6. Escritura de resultados

In [29]:
resumen = pd.DataFrame([
    ("proveedores (features)", feat_prov.count()), ("entidades (features)", feat_ent.count()),
    ("pares entidad × proveedor", nnz_ep), ("densidad entidad × proveedor", d_ep),
    ("pares entidad × familia", nnz_ef), ("densidad entidad × familia", d_ef),
    ("vértices bipartito", vertices_bip.count()), ("aristas bipartito", aristas_bip.count()),
    ("vértices co-participación", vertices_co.count()), ("aristas co-participación", aristas_co.count()),
    ("vértices co-participación sin hubs", vertices_co_sh.count()), ("aristas co-participación sin hubs", aristas_co_sh.count()),
    ("vértices KNN", vertices_knn.count()), ("aristas KNN", aristas_knn.count())],
    columns=["medida", "valor"])
escritos = guardar_tablas({"resumen_artefactos": resumen, "validacion_artefactos": validacion, "comparacion_grafos": comparacion,
                           "particiones_artefactos": particiones, "dag_self_join": dag,
                           "grado_coparticipacion": grado_co.groupBy(F.col("count").alias("grado")).agg(F.count("*").alias("nodos"))})
resumen.assign(valor=resumen.valor.map(lambda v: f"{v:.4%}" if v < 1 else f"{v:,.0f}"))

Escrito en 01_Trans/temp/c_uso/: resumen_artefactos, validacion_artefactos, comparacion_grafos, particiones_artefactos, dag_self_join, grado_coparticipacion


,medida,valor
0,proveedores (features),"86,032"
1,entidades (features),"2,996"
2,pares entidad × proveedor,"136,124"
3,densidad entidad × proveedor,0.0529%
4,pares entidad × familia,"39,349"
5,densidad entidad × familia,3.4071%
6,vértices bipartito,"89,024"
7,aristas bipartito,"136,124"
8,vértices co-participación,"60,230"
9,aristas co-participación,"1,640,718"


#### a. Verificación

In [30]:
verificar_salidas(escritos)

OK: 6 tablas (parquet + json)


---
### 7. Cerrar sesión

In [31]:
spark.stop()